# Rental Listing Data

This notebook loads rental listings from `data.json` into a pandas DataFrame for exploration and modeling.


## 1. Imports and configuration

Install dependencies if needed: `python -m pip install pandas scikit-learn matplotlib`, then install the warning-sign detectors from the repo root with `python -m pip install -e housing-fraud-nlp`.


In [13]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay,
                             balanced_accuracy_score, f1_score)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

from housing_fraud_nlp import WarningSignFeatures

RANDOM_STATE = 42
TEST_SIZE = 0.2
MAX_DEPTH = 3  # Small cap keeps the tree readable and limits overfitting


## 2. Load Data

Each JSON object represents one rental listing. The loader returns one DataFrame row per listing; `image_urls` stays as a list in each row.


In [14]:
def load_listings(json_path=Path.cwd() / "data.json"):
    """Read the listings JSON file into a pandas DataFrame."""
    return pd.read_json(json_path, encoding="utf-8")

data = load_listings()
data.head()


,title,description,price,currency,price_period,location_text,latitude,longitude,image_urls,source_url,Fake?
0,UBC Ponderosa Commons Studio with Sea View,Date: 8.10-9.7\nHigh-rise ocean view room for ...,800,CAD,month,Vancouver,49.264453,-123.256100,[https://scontent-sea5-1.xx.fbcdn.net/v/t39.84...,https://www.facebook.com/marketplace/item/1334...,False
1,Studio 1 Bath Apartment,**One month FREE RENT!!!!**\nBright Studio uni...,1950,CAD,month,Vancouver,NaN,NaN,[https://scontent-sea5-1.xx.fbcdn.net/v/t39.84...,https://www.facebook.com/marketplace/item/2105...,False
2,3 Beds 3.5 Baths House,"$4,500 / 3br - 1200ft2 - 3.5WR utilities INCLU...",4500,CAD,month,Vancouver,49.251061,-123.125316,[https://scontent-sea5-1.xx.fbcdn.net/v/t39.84...,https://www.facebook.com/marketplace/item/2179...,False
3,2 Bed 1 Bath - House,216X 49th Ave Vancouver Brand New Laneway Hous...,2200,CAD,month,Vancouver,49.224993,-123.063330,[https://scontent-sea5-1.xx.fbcdn.net/v/t39.30...,https://www.facebook.com/marketplace/item/1789...,False
4,2 Beds 1 Bath Apartment,"Clean, modern 2 bedroom suite on quiet street ...",2200,CAD,month,Vancouver,49.261860,-123.172776,[https://scontent-sea5-1.xx.fbcdn.net/v/t39.84...,https://www.facebook.com/marketplace/item/1044...,False


## 3. Prepare features and target

The JSON will include a boolean `Fake?` target. Use the requested listing details as model inputs. This accepts either `lon`/`lat` or the current JSON names `longitude`/`latitude`.


In [15]:
target_column = "Fake?"
interest_columns = ["price", "description", "currency", "price_period", "title", "lon", "lat"]

# Current JSON uses longitude/latitude; rename when the target data is available.
data = data.rename(columns={"longitude": "lon", "latitude": "lat"})
X = data[interest_columns].copy()
y = data[target_column].astype(bool)


## 4. Split into training and test sets

Stratification preserves the fake/real proportion in each split. Uncomment the code after defining `X` and `y`.


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")


Training rows: 16 | Test rows: 4


## 5. Define the decision tree pipeline

The pipeline preprocesses numeric, categorical, and text columns before fitting the tree. `WarningSignFeatures` runs the five rule-based warning-sign detectors on the title and description. Run after `X` and `y` have been prepared.


In [ ]:
numeric_columns = ["price", "lon", "lat"]
categorical_columns = ["currency", "price_period"]

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_columns),
        ("description", TfidfVectorizer(max_features=300, min_df=2), "description"),
        ("title", TfidfVectorizer(max_features=100, min_df=1), "title"),
        ("warning_signs", WarningSignFeatures(), ["title", "description"]),
    ],
    remainder="drop",
)

model = Pipeline([
    ("preprocess", preprocess),
    ("tree", DecisionTreeClassifier(
        max_depth=MAX_DEPTH, min_samples_leaf=3, class_weight="balanced",
        random_state=RANDOM_STATE
    )),
])

model.fit(X_train, y_train)
model.score(X_train, y_train)

1.0

### What the decision tree trains on

The preprocessing step turns each listing into one row of numbers. This shows that intermediate table for the training set. The TF-IDF columns (`description__*`, `title__*`) are word weights; the `warning_signs__*` columns are the detector flags (0/1) and evidence-sentence counts.

In [ ]:
fitted_preprocess = model.named_steps["preprocess"]
matrix = fitted_preprocess.transform(X_train)
matrix = matrix.toarray() if hasattr(matrix, "toarray") else matrix
tree_input = pd.DataFrame(matrix, columns=fitted_preprocess.get_feature_names_out(), index=X_train.index)

print(f"{tree_input.shape[0]} listings x {tree_input.shape[1]} features")
print(tree_input.columns.str.split("__").str[0].value_counts().to_string())

# Every non-TF-IDF column; the TF-IDF columns are too many to read.
tree_input.loc[:, ~tree_input.columns.str.startswith(("description__", "title__"))]

In [20]:
model.score(X_test, y_test)

1.0

## 6. Five-fold cross-validation

This reports balanced accuracy for each fold and its mean and standard deviation. Every fold fits its own preprocessing pipeline, avoiding text vocabulary leakage across folds.


In [18]:
CV_FOLDS = 5
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy")

print("Fold balanced accuracy:", cv_scores)
print(f"Mean balanced accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")


Fold balanced accuracy: [1. 1. 1. 1. 1.]
Mean balanced accuracy: 1.000 +/- 0.000


C:\Users\Rodrigo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\Rodrigo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\Rodrigo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion mat